<a href="https://colab.research.google.com/github/LarsVoermans/master-thesis-pead/blob/main/PreProcessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Dataset opnieuw inladen
Financial_dataset = pd.read_csv("Offical_Dataset.csv")

In [ ]:
#Dropping of dublicates
Financial_dataset = Financial_dataset.drop_duplicates(
    subset=["Ticker", "Date"],
    keep="first"
)

In [ ]:
# Making Return_class
conditions = [
    (Financial_dataset['Return'] > 5) & (Financial_dataset['Return'] <= 10),
    (Financial_dataset['Return'] > 10),
    (Financial_dataset['Return'] >= -5) & (Financial_dataset['Return'] <= 5),
    (Financial_dataset['Return'] < -5) & (Financial_dataset['Return'] >= -10),
    (Financial_dataset['Return'] < -10)
]

choices = [
    1,
    2,
    0,
    -1,
    -2
]

Financial_dataset['Return_class'] = np.select(conditions, choices, default=np.nan)

Financial_dataset['Return_class'] = np.select(
    conditions,
    choices,
    default=np.nan
)



In [ ]:
#Splitting test en train/val

# Year is integer
Financial_dataset['Year'] = Financial_dataset['Year'].astype(int)

# Test set = everything from 2024
test = Financial_dataset[Financial_dataset['Year'] == 2024]

# Train + Validation = everything before 2024
train_val = Financial_dataset[Financial_dataset['Year'] < 2024]

# Check sizes
print("Train + Val size:", len(train_val))
print("Test size:", len(test))

Train + Val size: 11271
Test size: 1292


In [ ]:
#Splitting train and validation on ticker level

#Grouping the dataset on ticker level
groups = train_val.groupby('Ticker')
tickers = list(groups.groups.keys())

#SHuffle the dataset so that everything is random on ticker level
np.random.shuffle(tickers)

#making a split on 80%
split = int(len(tickers) * 0.8)

train_tickers = tickers[:split]
val_tickers = tickers[split:]

train = train_val[train_val['Ticker'].isin(train_tickers)]
val = train_val[train_val['Ticker'].isin(val_tickers)]

In [ ]:
#Co
print("Train distribution")
print(train['Return_class'].value_counts(normalize=True))

print("\nValidation distribution")
print(val['Return_class'].value_counts(normalize=True))

print("\nTest distribution")
print(test['Return_class'].value_counts(normalize=True))

Train distribution
Return_class
 0.0    0.772486
 1.0    0.066460
-1.0    0.059149
 2.0    0.052282
-2.0    0.049623
Name: proportion, dtype: float64

Validation distribution
Return_class
 0.0    0.846188
 1.0    0.050825
-1.0    0.043246
 2.0    0.030762
-2.0    0.028979
Name: proportion, dtype: float64

Test distribution
Return_class
 0.0    0.561146
 2.0    0.132353
-2.0    0.126935
 1.0    0.099071
-1.0    0.080495
Name: proportion, dtype: float64


In [ ]:
!pip install pyarrow

train.to_parquet("train_preprocessed.parquet")
val.to_parquet("val_preprocessed.parquet")
test.to_parquet("test_preprocessed.parquet")